# Multi-Agent Architecture Walkthrough (Phase 2)

This notebook demonstrates how we migrate away from a simple black-box `create_react_agent` and build a customized, highly controllable **StateGraph** with specialized node agents: **Planner**, **Executor**, and **Reviewer**.

## 1. The Global State Schema
First, we define a richer state. Instead of just passing `messages` back and forth, we give the graph formal memory to hold an execution plan and track its progress.

In [ ]:
from typing import Annotated, List, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class MultiAgentState(TypedDict):
    # Standard message history (appends new messages from nodes)
    messages: Annotated[list, add_messages]
    
    # The chronological execution plan generated by the Planner
    plan: List[str]
    
    # Tracking execution state
    current_step: int
    
    # The output condition from the Reviewer node
    review_status: Literal["pending", "approved", "rejected"]

## 2. Defining the Node Functions
Each node is just a Python function that takes the current `MultiAgentState`, does some AI reasoning, and returns a dict with the updated fields.

In [ ]:
# ---------------------------------------------------------
# NODE 1: The Planner
# Analyzes request, outputs a strict list of steps.
# ---------------------------------------------------------
def planner_node(state: MultiAgentState):
    print("\n[ 🧠 Planner Node ] Analyzing request...")
    # In reality, this would be an LLM call with structured output.
    # We mock it here based on the user's message.
    user_request = state["messages"][0][1]
    
    plan = [
        "1. Run 'apt-get update'",
        "2. Run 'apt-get install nginx -y'",
        "3. Run 'systemctl status nginx' to verify"
    ]
    print(f"   --> Generated Plan: {plan}")
    return {"plan": plan, "current_step": 0, "review_status": "pending"}

# ---------------------------------------------------------
# NODE 2: The Executor
# Runs exactly one step of the plan, logs output, increments step.
# ---------------------------------------------------------
def executor_node(state: MultiAgentState):
    step_idx = state.get("current_step", 0)
    plan = state.get("plan", [])
    
    if step_idx >= len(plan):
        return {} # Nothing left to execute
        
    current_task = plan[step_idx]
    print(f"\n[ ⚙️ Executor Node ] Executing step {step_idx + 1}/{len(plan)}: {current_task}")
    
    # MOCK API/SSH Execution
    print(f"   --> [SSH Output]: Success. Done.")
    
    # Return updated state: append to system messages, increment step forward
    return {
        "messages": [("system", f"Action Output for '{current_task}': Success")],
        "current_step": step_idx + 1
    }

# ---------------------------------------------------------
# NODE 3: The Reviewer
# Looks at the Executor's work, decides if we succeeded.
# ---------------------------------------------------------
def reviewer_node(state: MultiAgentState):
    print("\n[ 🧐 Reviewer Node ] Inspecting final outputs...")
    
    # Mock Review Logic: If executor completed all steps, we approve.
    # In reality, an LLM checks if stdout says "failed" or "error".
    print("   --> Evaluation: All tasks completed successfully. Architecture matches goal.")
    return {
        "review_status": "approved",
        "messages": [("ai", "I have successfully completed your deployment request!")]
    }

## 3. Assembling the StateGraph & Routers
We use conditional routing to dictate how the graph flows based on the state.

In [ ]:
from langgraph.graph import StateGraph, START, END

# ROUTER: Does Executor have more steps, or should we go to Reviewer?
def route_after_execution(state: MultiAgentState):
    if state.get("current_step", 0) < len(state.get("plan", [])):
        return "executor"  # Loop back into executor
    else:
        return "reviewer"  # All steps done, time to review

# ROUTER: Did the Reviewer approve, or send back to Planner?
def route_after_review(state: MultiAgentState):
    if state.get("review_status") == "approved":
        return END
    else:
        return "planner"


# Compile Graph!
workflow = StateGraph(MultiAgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)
workflow.add_node("reviewer", reviewer_node)

# Edges: The logic pathways
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "executor")

# Conditional Edges (Routers)
workflow.add_conditional_edges(
    "executor", 
    route_after_execution, 
    {"executor": "executor", "reviewer": "reviewer"}
)

workflow.add_conditional_edges(
    "reviewer", 
    route_after_review, 
    {"planner": "planner", END: END}
)

app = workflow.compile()

## 4. Run the Simulation
Watch how execution safely jumps between our 3 specialized nodes.

In [ ]:
print("\n--- 🚀 IGNITING MULTI-AGENT PIPELINE --- ")
initial_state = {
    "messages": [("user", "Install and verify Nginx on the server")]
}

for event in app.stream(initial_state):
    # This loops and prints updates as each node finishes its execution
    pass

print("\n--- ✅ PIPELINE FINISHED ---")
